# Notebook 03 — Funciones de fechas

Tercer sub-bloque del Tema 05. Las **funciones de fecha** son las que más usas en BI: agrupar por mes/trimestre/año, calcular antigüedades, comparar períodos, detectar días no laborables. PostgreSQL trae un set muy completo — vas a ver los esenciales aquí.

Cubres: `NOW`/`CURRENT_DATE`/`CURRENT_TIMESTAMP`, `DATE_TRUNC` para agregación temporal, `EXTRACT` para sacar componentes, aritmética con `INTERVAL`, diferencias con `AGE`, y formateo con `TO_CHAR`/`TO_DATE`.

Cierre del notebook: query analítica comparando el mismo análisis temporal **con** y **sin** `dim_date` — para reforzar el valor del modelo dimensional que viste en el Tema 02.

**Contenido de este notebook:**

- [Setup](#setup)
- [Tipos de fecha en PostgreSQL](#tipos-de-fecha-en-postgresql)
- [`NOW`, `CURRENT_DATE`, `CURRENT_TIMESTAMP` — el ahora](#now-current_date-current_timestamp--el-ahora)
- [`DATE_TRUNC` — la herramienta #1 para agregar por período](#date_trunc--la-herramienta-1-para-agregar-por-período)
- [`EXTRACT` — sacar un componente específico](#extract--sacar-un-componente-específico)
- [`INTERVAL` — aritmética con fechas](#interval--aritmética-con-fechas)
- [`AGE` — diferencia legible entre dos fechas](#age--diferencia-legible-entre-dos-fechas)
- [`TO_CHAR` y `TO_DATE` — formateo y parseo](#to_char-y-to_date--formateo-y-parseo)
- [Caso integrador — `dim_date` justifica su existencia](#caso-integrador--dim_date-justifica-su-existencia)

## Setup

In [ ]:
%load_ext sql

from sqlalchemy import create_engine

AURORA_HOST     = "aurora-mod4.cluster-xxxxx.us-east-1.rds.amazonaws.com"
AURORA_PASSWORD = "TU_PASSWORD_AQUI"
AURORA_DATABASE = "northwind"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{AURORA_PASSWORD}@{AURORA_HOST}:5432/{AURORA_DATABASE}"
)

%sql engine

## Tipos de fecha en PostgreSQL

Antes de usar las funciones, ten claros los cuatro tipos básicos:

| Tipo | Qué guarda | Ejemplo |
|---|---|---|
| `DATE` | Solo fecha (sin hora) | `2024-05-15` |
| `TIME` | Solo hora (sin fecha) | `14:30:00` |
| `TIMESTAMP` | Fecha + hora **sin** zona horaria | `2024-05-15 14:30:00` |
| `TIMESTAMPTZ` | Fecha + hora **con** zona horaria | `2024-05-15 14:30:00-06:00` |
| `INTERVAL` | Una **duración** (no un instante) | `CAST('2 days 3 hours' AS INTERVAL)` |

En Northwind las fechas son todas `DATE` (sin hora). En logs de aplicaciones reales, casi siempre vas a ver `TIMESTAMPTZ` — la zona horaria importa cuando los usuarios están en países distintos.

## `NOW`, `CURRENT_DATE`, `CURRENT_TIMESTAMP` — el ahora

Tres funciones que devuelven el momento actual, pero con tipo distinto:

| Función | Tipo devuelto | Equivalente |
|---|---|---|
| `NOW()` | `TIMESTAMPTZ` | `CURRENT_TIMESTAMP` |
| `CURRENT_TIMESTAMP` | `TIMESTAMPTZ` | `NOW()` |
| `CURRENT_DATE` | `DATE` (sin hora) | — |
| `CURRENT_TIME` | `TIME` (solo hora) | — |
| `LOCALTIMESTAMP` | `TIMESTAMP` (sin tz) | — |

In [ ]:
%%sql
SELECT
    NOW()               AS now,
    CURRENT_DATE        AS current_date,
    CURRENT_TIME        AS current_time,
    CURRENT_TIMESTAMP   AS current_timestamp,
    LOCALTIMESTAMP      AS localtimestamp;

**Punto sutil pero importante:** dentro de una **misma transacción**, todas estas funciones devuelven exactamente el mismo valor — el instante en que **empezó la transacción**. Si quieres el reloj real momento-a-momento, usa `CLOCK_TIMESTAMP()` (que sí avanza dentro de la misma transacción).

Eso es deliberado: en SQL, si calculas `NOW() - order_date` en distintas filas de una query, quieres que **todas** se calculen contra el mismo instante de referencia.

## `DATE_TRUNC` — la herramienta #1 para agregar por período

`DATE_TRUNC('unidad', fecha)` **trunca** la fecha al inicio de la unidad indicada. Es como redondeo, pero siempre hacia abajo.

Unidades válidas: `'microseconds'`, `'milliseconds'`, `'second'`, `'minute'`, `'hour'`, `'day'`, `'week'`, `'month'`, `'quarter'`, `'year'`, `'decade'`, `'century'`, `'millennium'`.

In [ ]:
%%sql
SELECT
    CAST('1996-07-15' AS DATE)                     AS original,
    DATE_TRUNC('day',     CAST('1996-07-15 14:30:00' AS TIMESTAMP)) AS truncado_dia,
    DATE_TRUNC('month',   CAST('1996-07-15' AS DATE)) AS truncado_mes,
    DATE_TRUNC('quarter', CAST('1996-07-15' AS DATE)) AS truncado_trimestre,
    DATE_TRUNC('year',    CAST('1996-07-15' AS DATE)) AS truncado_año,
    DATE_TRUNC('week',    CAST('1996-07-15' AS DATE)) AS truncado_semana;

`DATE_TRUNC('month', ...)` devuelve el **primer día** del mes (`1996-07-01`). Por eso es perfecto para `GROUP BY`: todas las fechas de julio caen en el mismo grupo `1996-07-01`.

Ejemplo clásico: ventas mensuales sobre `fact_sales`:

In [ ]:
%%sql
SELECT
    CAST(DATE_TRUNC('month', dd.full_date) AS DATE)  AS mes,
    COUNT(*)                                 AS lineas,
    ROUND(SUM(fs.line_total), 2)             AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date dd ON dd.date_key = fs.order_date_key
GROUP BY  DATE_TRUNC('month', dd.full_date)
ORDER BY  mes;

**`DATE_TRUNC('week', ...)` en PostgreSQL trunca al lunes** (ISO 8601). Si necesitas que la semana empiece en domingo (convención EE.UU.), tienes que jugar con `DATE_TRUNC` + `INTERVAL` o usar `dim_date.week_of_year` directamente.

## `EXTRACT` — sacar un componente específico

`EXTRACT(unidad FROM fecha)` devuelve **un número** representando esa unidad. Útil cuando quieres el **valor** del componente (4 para abril), no la fecha truncada.

Diferencia clave con `DATE_TRUNC`:

- `DATE_TRUNC('month', '1996-07-15')` → `1996-07-01` (sigue siendo fecha)
- `EXTRACT(month FROM '1996-07-15')` → `7` (es un número)

Unidades comunes: `year`, `quarter`, `month`, `week`, `day`, `dow` (día de la semana, 0=domingo), `isodow` (1=lunes, ISO), `hour`, `minute`, `second`, `epoch` (segundos desde 1970).

In [ ]:
%%sql
SELECT
    CAST('1996-07-15' AS DATE)                    AS fecha,
    EXTRACT(year    FROM CAST('1996-07-15' AS DATE)) AS año,
    EXTRACT(quarter FROM CAST('1996-07-15' AS DATE)) AS trimestre,
    EXTRACT(month   FROM CAST('1996-07-15' AS DATE)) AS mes,
    EXTRACT(week    FROM CAST('1996-07-15' AS DATE)) AS semana_iso,
    EXTRACT(day     FROM CAST('1996-07-15' AS DATE)) AS dia,
    EXTRACT(dow     FROM CAST('1996-07-15' AS DATE)) AS dow_dom_0,
    EXTRACT(isodow  FROM CAST('1996-07-15' AS DATE)) AS isodow_lun_1;

**Truco mnemotécnico:** si vas a **agrupar** una serie de fechas, usa `DATE_TRUNC` (todas las fechas del mismo período colapsan al mismo timestamp). Si vas a **etiquetar/filtrar** por componente, usa `EXTRACT`.

```sql
GROUP BY  DATE_TRUNC('quarter', fecha)        -- agrupar fechas del mismo trimestre
WHERE     EXTRACT(year FROM fecha) = 1997     -- filtrar las del año 1997
```

In [ ]:
%%sql
-- Ventas por trimestre y día de la semana (¿qué día se vende más?)
SELECT
    CAST(EXTRACT(isodow FROM dd.full_date) AS INT)  AS dow_iso,
    dd.day_of_week_name,
    COUNT(*)                                AS lineas,
    ROUND(SUM(fs.line_total), 2)            AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date   dd ON dd.date_key = fs.order_date_key
GROUP BY  EXTRACT(isodow FROM dd.full_date), dd.day_of_week_name
ORDER BY  dow_iso;

## `INTERVAL` — aritmética con fechas

`INTERVAL` representa una **duración**, no un instante. Soporta sintaxis de texto muy expresiva:

```sql
INTERVAL '1 day'
INTERVAL '3 days 4 hours 30 minutes'
INTERVAL '2 months'
INTERVAL '1 year 6 months'
```

Y se combina con fechas con los operadores habituales `+` y `-`:

In [ ]:
%%sql
SELECT
    CAST('1996-07-15' AS DATE)                                 AS original,
    CAST('1996-07-15' AS DATE) + INTERVAL '7 days'             AS mas_una_semana,
    CAST('1996-07-15' AS DATE) - INTERVAL '1 month'            AS menos_un_mes,
    CAST('1996-07-15' AS DATE) + INTERVAL '1 year 2 months'    AS mas_un_año_dos_meses,
    NOW() - INTERVAL '24 hours'                        AS hace_24_horas;

Caso típico en BI: **filtrar por ventana móvil de tiempo**. "Pedidos de los últimos N días" se expresa naturalmente con `INTERVAL`:

In [ ]:
%%sql
-- Pedidos en los 30 días anteriores al último pedido registrado
WITH ult AS (
    SELECT MAX(full_date) AS hasta FROM northwind_dwh.dim_date
    JOIN northwind_dwh.fact_sales fs ON fs.order_date_key = dim_date.date_key
)
SELECT
    dd.full_date,
    COUNT(*)                       AS lineas,
    ROUND(SUM(fs.line_total), 2)   AS ventas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date dd ON dd.date_key = fs.order_date_key
WHERE     dd.full_date >= (SELECT hasta - INTERVAL '30 days' FROM ult)
GROUP BY  dd.full_date
ORDER BY  dd.full_date DESC
LIMIT 10;

**Resta directa entre fechas:** `fecha1 - fecha2` devuelve un `INTERVAL` (si son `TIMESTAMP`s) o un **número de días** (si son `DATE`s).

In [ ]:
%%sql
-- Días que tardó cada pedido entre order_date y shipped_date
SELECT
    fs.order_id,
    d_ord.full_date     AS fecha_orden,
    d_shp.full_date     AS fecha_envio,
    d_shp.full_date - d_ord.full_date AS dias_para_envio
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date d_ord ON d_ord.date_key = fs.order_date_key
LEFT JOIN northwind_dwh.dim_date d_shp ON d_shp.date_key = fs.shipped_date_key
WHERE     d_shp.full_date IS NOT NULL
ORDER BY  dias_para_envio DESC
LIMIT 10;

**Patrón Kimball en acción:** `d_ord` y `d_shp` son los dos *aliases* de `dim_date` jugando los roles de "fecha de orden" y "fecha de envío" — exactamente lo que viste en el Tema 02 como **role-playing dimension**.

## `AGE` — diferencia legible entre dos fechas

`AGE(fecha1, fecha2)` devuelve un `INTERVAL` en términos humanos (años, meses, días) en vez del total raw en días. Útil cuando quieres reportar antigüedades sin tener que dividir y truncar.

In [ ]:
%%sql
SELECT
    full_name,
    hire_date,
    CURRENT_DATE - hire_date     AS dias_en_empresa,
    AGE(CURRENT_DATE, hire_date) AS antiguedad_legible
FROM     northwind_dwh.dim_employee
ORDER BY hire_date;

`AGE(x, y)` con un solo argumento es equivalente a `AGE(CURRENT_DATE, x)` — ahorra escribir el `CURRENT_DATE`.

**Caveat:** los empleados de Northwind fueron contratados entre 1992 y 1994. Con `CURRENT_DATE` real del 2026, la antigüedad mostrada será gigante (~33 años) — eso es porque el dataset es ficticio, no porque la función esté mal.

## `TO_CHAR` y `TO_DATE` — formateo y parseo

Estas dos son traductoras entre **fecha** y **string** con formato custom:

- **`TO_CHAR(fecha, 'formato')`** — fecha → string.
- **`TO_DATE(string, 'formato')`** — string → `DATE`.

Patrones útiles del formato:

| Patrón | Significado | Ejemplo |
|---|---|---|
| `YYYY` | Año (4 dígitos) | `1996` |
| `MM` | Mes (2 dígitos) | `07` |
| `DD` | Día del mes | `15` |
| `HH24:MI:SS` | Hora, minuto, segundo | `14:30:00` |
| `Month` | Mes en inglés | `July` |
| `Dy` | Día abreviado | `Mon` |
| `Q` | Trimestre | `3` |
| `IW` | Semana ISO | `29` |

In [ ]:
%%sql
SELECT
    CAST('1996-07-15' AS DATE)                                       AS original,
    TO_CHAR(CAST('1996-07-15' AS DATE), 'YYYYMMDD')                  AS smart_key,
    TO_CHAR(CAST('1996-07-15' AS DATE), 'DD/MM/YYYY')                AS formato_es,
    TO_CHAR(CAST('1996-07-15' AS DATE), 'Day, DD "de" Month "de" YYYY') AS legible_en,
    TO_CHAR(CAST('1996-07-15' AS DATE), '"Q"Q YYYY')                 AS trimestre_label,
    TO_DATE('15/07/1996', 'DD/MM/YYYY')                      AS parseado;

**`CAST(TO_CHAR(fecha, 'YYYYMMDD') AS INT)` es exactamente cómo se generó la smart key de `dim_date`** en el script `02_dim_date_populate.sql` del Tema 02. Y `TO_DATE` es lo opuesto, útil cuando recibes fechas en formato de texto custom desde una fuente externa.

## Caso integrador — `dim_date` justifica su existencia

Mismo análisis (ventas por trimestre, solo días laborables) escrito **con** y **sin** `dim_date`. El contraste es el argumento más fuerte para el modelo dimensional.

**Versión sin `dim_date`** — calculas todo on-the-fly desde la fecha cruda:

In [ ]:
%%sql
-- Hipotético: si fact_sales tuviera order_date como DATE en vez de date_key
-- Aquí lo simulamos joinando dim_date para obtener la fecha cruda
SELECT
    CAST(EXTRACT(year    FROM dd.full_date) AS INT)  AS año,
    CAST(EXTRACT(quarter FROM dd.full_date) AS INT)  AS trimestre,
    COUNT(*)                                 AS lineas,
    ROUND(SUM(fs.line_total), 2)             AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date   dd ON dd.date_key = fs.order_date_key
WHERE     EXTRACT(isodow FROM dd.full_date) NOT IN (6, 7)   -- excluir sábado y domingo
GROUP BY  EXTRACT(year FROM dd.full_date), EXTRACT(quarter FROM dd.full_date)
ORDER BY  año, trimestre;

**Versión con `dim_date`** — usa las columnas pre-calculadas:

In [ ]:
%%sql
SELECT
    dd.year     AS año,
    dd.quarter  AS trimestre,
    COUNT(*)                       AS lineas,
    ROUND(SUM(fs.line_total), 2)   AS ventas_netas
FROM      northwind_dwh.fact_sales fs
JOIN      northwind_dwh.dim_date   dd ON dd.date_key = fs.order_date_key
WHERE     NOT dd.is_weekend            -- columna pre-calculada
GROUP BY  dd.year, dd.quarter
ORDER BY  dd.year, dd.quarter;

Comparación:

| Aspecto | Sin `dim_date` | Con `dim_date` |
|---|---|---|
| Legibilidad | `EXTRACT(year FROM dd.full_date)` × 2 | `dd.year` |
| Días no laborables | `EXTRACT(isodow ...) NOT IN (6, 7)` | `NOT dd.is_weekend` |
| Festivos / fin de mes / inicio de Q | Imposible sin agregar lógica | Columnas que puedes agregar a `dim_date` una sola vez |
| Performance | Recalcula EXTRACT por cada fila | Lectura directa de columna indexable |

Ese **"recalcula por cada fila"** es el costo escondido: en una fact de mil millones de filas, calcular `EXTRACT(isodow ...)` 1B veces es notablemente más lento que leer `is_weekend` directo. `dim_date` materializa el cálculo **una vez** (al poblarse) y lo amortiza para siempre.

## Cierre

Lo que cubriste:

| Tema | Funciones / patrones |
|---|---|
| Tipos de fecha | `DATE`, `TIME`, `TIMESTAMP`, `TIMESTAMPTZ`, `INTERVAL` |
| Momento actual | `NOW`, `CURRENT_DATE`, `CURRENT_TIMESTAMP`, `LOCALTIMESTAMP` |
| Truncar para agrupar | `DATE_TRUNC('mes', fecha)` |
| Extraer un componente | `EXTRACT(year FROM fecha)`, `EXTRACT(isodow FROM fecha)` |
| Aritmética con duraciones | `fecha + INTERVAL '7 days'`, restas directas |
| Diferencia legible | `AGE(fecha)` |
| Formateo / parseo | `TO_CHAR(fecha, 'YYYYMMDD')`, `TO_DATE('15/07/1996', 'DD/MM/YYYY')` |
| Role-playing | Dos aliases de `dim_date` en la misma query |

**Cierre del bloque conceptual:** ya conoces los tres tipos de funciones predefinidas más usadas en BI — agregadas, de strings, de fechas. El último notebook (**04 — Práctica**) te da 15 ejercicios graduales que combinan los tres tipos, sobre Northwind DWH y Airbnb. Es donde se consolidan los reflejos.

---

<p align="center">
<a href="02_funciones_de_strings.ipynb">← Anterior: Notebook 02</a> | <a href="Readme.md">Volver al índice</a> | <a href="04_practica.ipynb">Siguiente: Notebook 04 — Práctica →</a>
</p>